# Tuning: the regularization sweep

`C` is the one knob that matters for this linear model — same job repeated
with different `SM_HPS`, scored by each run's `metrics.json`.

In [ ]:
import json

import boto3
import pandas as pd
from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession

session = LocalSession()
BUCKET = "wine-quality-classification"
s3 = boto3.client(
    "s3", endpoint_url="http://wine-quality-classification-moto:5000"
)
rows = []
for c in [0.01, 0.1, 1.0, 10.0]:
    est = Estimator(
        image_uri="wine-quality-classification:train",
        role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
        instance_count=1,
        instance_type="local",
        output_path=f"s3://{BUCKET}/output",
        sagemaker_session=session,
        hyperparameters={"C": c, "max_iter": 2000},
    )
    est.set_entry_point(
        source_dir="src/wine_quality_classification", entry_point="train.py"
    )
    est.fit({"train": f"s3://{BUCKET}/input/train"})
    m = json.loads(
        s3.get_object(
            Bucket=BUCKET,
            Key=f"output/{est.latest_training_job.name}/output/metrics.json",
        )["Body"].read()
    )
    rows.append({"C": c, **m})
pd.DataFrame(rows).sort_values("accuracy", ascending=False)